<a href="https://colab.research.google.com/github/eltongaspar/python/blob/Advpl/GROQ_C%C3%93DIGO_COMPLETO_%E2%80%93_MULTI_AGENTES_PERSONALIZ%C3%81VEIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gradio groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 2.9 MB/s eta 0:00:00


In [5]:
import gradio as gr
from groq import Groq

# ---------- SUA API KEY ----------
GROQ_API_KEY = ""
# ---------------------------------

client = Groq(api_key=GROQ_API_KEY)


# ✅ Função para gerar resposta de um agente
def gerar_resposta(modelo, personalidade, mensagens):
    try:
        resposta = client.chat.completions.create(
            model=modelo,
            messages=[
                {"role": "system", "content": personalidade},
                *mensagens
            ]
        )
        return resposta.choices[0].message.content
    except Exception as e:
        return f"⚠️ ERRO NO MODELO: {str(e)}"


# ✅ Função principal: supervisor envia e IA responde
def chat(supervisor_msg, cliente_pers, atendente_pers, historico):

    mensagens = []
    for user, ia in historico:
        mensagens.append({"role": "user", "content": user})
        mensagens.append({"role": "assistant", "content": ia})

    # Adiciona mensagem do supervisor
    mensagens.append({"role": "user", "content": supervisor_msg})

    # ✅ Cliente bravo/informado/etc.
    resposta_cliente = gerar_resposta(
        modelo="llama-3.1-8b-instant",
        personalidade=cliente_pers,
        mensagens=mensagens
    )

    # ✅ Atendente inexperiente/calmo/etc.
    resposta_atendente = gerar_resposta(
        modelo="llama-3.1-8b-instant",
        personalidade=atendente_pers,
        mensagens=[
            {"role": "user", "content": supervisor_msg},
            {"role": "assistant", "content": resposta_cliente}
        ]
    )

    resposta_final = (
        f"### 👨‍💼 Supervisor (Você)\n{supervisor_msg}\n\n"
        f"### 😡 Cliente\n{resposta_cliente}\n\n"
        f"### 🧑‍💻 Atendente\n{resposta_atendente}"
    )

    historico.append((supervisor_msg, resposta_final))

    return "", historico


# ✅ Interface
with gr.Blocks() as demo:
    gr.Markdown("## 🎭 Sistema Multiagentes – Cliente, Atendente e Supervisor (Você)")

    with gr.Row():
        cliente_p = gr.Textbox(
            label="Personalidade do Cliente",
            placeholder="Ex.: Você é um cliente bravo, impaciente e reclama de tudo...",
            lines=4
        )
        atendente_p = gr.Textbox(
            label="Personalidade do Atendente",
            placeholder="Ex.: Você é um atendente inexperiente, inseguro e comete erros...",
            lines=4
        )

    chatbot = gr.Chatbot(height=450)
    supervisor_input = gr.Textbox(
        placeholder="Digite sua mensagem como SUPERVISOR...",
        label="Mensagem do Supervisor"
    )

    supervisor_input.submit(
        chat,
        [supervisor_input, cliente_p, atendente_p, chatbot],
        [supervisor_input, chatbot]
    )

demo.launch()

/tmp/ipykernel_1404/3165573973.py:81: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=450)
/tmp/ipykernel_1404/3165573973.py:81: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=450)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4d3495e8ef8f8256a2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
